In [ ]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio
pio.renderers.default='notebook'
df = pd.read_csv("WeatherEvents_Jan2016-Dec2022.csv")

In [ ]:
df.columns

In [ ]:
print(df['Type'].unique())
print(df['Severity'].unique())

In [ ]:
#Selezionare eventi atmosferici storici che sono avvenuti nell'arco del tempo del nostro dataset e mostrarli con dei grafici (magari lo spostamento di un huragano (grafico animato))

#Avendo data di inizio e data di fine posso cercare gli eventi piu lunghi e vedere cosa sono

#Grafico HeatMap che mostra l'andamento degli eventi 'SEVERE' nei vari mesi per ogni anno (sulle X i mesi, sulle Y gli anni) il colore indica se ce ne sono stati tanti o pochi

#Trasformo la colonna StartTime in una data con pandas
df['StartTime(UTC)'] = pd.to_datetime(df['StartTime(UTC)'])

#Prendo solo quelli severi
df_severe = df[df['Severity'] == 'Severe'].copy()
#il copy va usato perche pandas ritorna una vista del dataseto originale se dopo modifichiamo df_severe potrebbe dare errore

df_severe['Year'] = df_severe['StartTime(UTC)'].dt.year
df_severe['Month'] = df_severe['StartTime(UTC)'].dt.month
#trasformandolo in datetime di pandas ci possiamo accedere con dt e abbiamo le sue funzionalità

#Creiamo la heatmap

#Contiamo per ongi combinazione di mese-anno il numero di eventi severe (collassa tutte le righe Gennaio 2016 in una sola contando il numero di occorrenze)
heatmap = df_severe.groupby(['Year','Month']).size().reset_index(name='Count')
#Quindi dopo qui abbiamo le righe singole che sono Gennaio 2016 / febbraio 2016 ... agosto 2016 e per ognuna di queste righe uniche abbiamo il numero di count 

#Creiamo una tabella pivot da passare alla heatmap
pivot_table = heatmap.pivot(index='Year', columns='Month', values='Count').fillna(0)

fig = px.imshow(pivot_table, 
                title="Stagionalità e Frequenza degli Eventi Estremi (2016-2022)",
                labels=dict(x="Mese", y="Anno", color="N eventi Estremi"),
                x=['Gen', 'Feb', 'Mar', 'Apr', 'Mag', 'Giu', 'Lug', 'Ago', 'Set', 'Ott', 'Nov', 'Dic']
                );

fig.show()


## Uragano Harvey 1

In [ ]:
#Analisi Uragano Harvey
df['StartTime(UTC)'] = pd.to_datetime(df['StartTime(UTC)'])

df_texas = df[df['State'] == 'TX'].copy()
#Facciamo la copy per lo stesso motivo di prima

df_texas['Year'] = df_texas['StartTime(UTC)'].dt.year
df_texas['Month'] = df_texas['StartTime(UTC)'].dt.month
df_texas['Day'] = df_texas['StartTime(UTC)'].dt.day

df_texas = df_texas[(df_texas['Year']==2017) & (df_texas['Month'] == 8)]

df_prec = df_texas[df_texas['Precipitation(in)'] > 0]

df_prec['Precipitation(mm)'] = df_prec['Precipitation(in)'] * 25.4;

fig = px.box(df_prec,
            x='Day',
            y='Precipitation(mm)',
            title="Distribuzione delle Precipitazioni Giornaliere in Texas - Uragano Harvey (Agosto 2017)",
            labels={'Day': 'Giorno', 'Precipitation(in)': 'Precipitazioni (pollici)'},
            )
fig.update_layout(xaxis=dict(tickmode='linear', dtick=1))

fig.add_vline(
    x=27,                           # Posizione sull'asse X (27 agosto)
    line_width=2,                   # Spessore della linea
    line_dash="dash",               # Stile della linea (tratteggiata)
    line_color="red",               # Colore della linea
    annotation_text="Picco Harvey (27 Ago)", # Testo dell'etichetta
    annotation_position="top left", # Posizione del testo
    annotation_font_color="red",
    opacity=0.5
)


fig.show()



In [ ]:
# 1. Filtra per Agosto 2017
df['StartTime(UTC)'] = pd.to_datetime(df['StartTime(UTC)'])
df_agosto_2017 = df[(df['StartTime(UTC)'].dt.year == 2017) & 
                    (df['StartTime(UTC)'].dt.month == 8) & 
                    (df['Precipitation(in)'] > 0)].copy()

# 2. Converti in millimetri
df_agosto_2017['Precipitation(mm)'] = df_agosto_2017['Precipitation(in)'] * 25.4

#Anomalia trovata nello stato di NY 1500mm (Impossibile) e 520mm (Impossibile), ho controllato nessun evento rilevante quel giorno

# 3. Abbassa la soglia di pulizia per escludere l'ulteriore anomalia di 520 mm
df_agosto_2017 = df_agosto_2017[df_agosto_2017['Precipitation(mm)'] <= 450]

# 4. Trova il PICCO MASSIMO per ogni Stato sui dati puliti
precipitazioni_stato = df_agosto_2017.groupby('State')['Precipitation(mm)'].max().reset_index()

# 5. Genera la Mappa Coropletica
fig = px.choropleth(
    precipitazioni_stato,
    locations='State',                 
    locationmode='USA-states',         
    color='Precipitation(mm)',         
    color_continuous_scale='Reds',
    range_color=[0, 430],              # Forza la scala dei colori per non farsi sfasare da eventuali altri piccoli outlier
    scope='usa',                       
    title="Picco Massimo di Precipitazioni per Stato negli USA (Agosto 2017)",
    labels={'Precipitation(mm)': 'Precipitazione Massima (mm)'}
)

fig.show()

In [ ]:
df['StartTime(UTC)'] = pd.to_datetime(df['StartTime(UTC)'])
df['EndTime(UTC)'] = pd.to_datetime(df['EndTime(UTC)'])

df_texas = df[(df['State'] == 'TX') & 
              (df['Type'] == 'Storm')]

display(df_texas)

## Uragano Harvey (Agosto 2017)

In [ ]:
df['StartTime(UTC)'] = pd.to_datetime(df['StartTime(UTC)'])

#Filtriamo solo per il mese di agosto del 2017
df_agosto_2017 = df[(df['StartTime(UTC)'].dt.year == 2017) & (df['StartTime(UTC)'].dt.month == 8)]
df_agosto_2017 = df_agosto_2017[df_agosto_2017['Severity'] == 'Heavy' ]

#Creiamo una colonna giorno per poter scorrere durante il mese dove si sposta l'uragano
df_agosto_2017['Day'] = df_agosto_2017['StartTime(UTC)'].dt.day

#Pulizia del dataset
df_agosto_2017['Precipitation(mm)'] = df_agosto_2017['Precipitation(in)'] * 25.4
df_agosto_2017 = df_agosto_2017[(df_agosto_2017['Precipitation(mm)'] >= 50) & (df_agosto_2017['Precipitation(mm)'] < 500) & (df_agosto_2017['Type'] == 'Rain')]
df_agosto_2017 = df_agosto_2017[df_agosto_2017['State'] != 'NY']

df_agosto_2017 = df_agosto_2017.sort_values(by='Day')

#display(df_agosto_2017)

fig = px.scatter_geo(
    df_agosto_2017,
    lat="LocationLat",
    lon="LocationLng",
    color="Precipitation(mm)",        # Il colore indica l'intensità
    size="Precipitation(mm)",         # La dimensione della bolla indica l'intensità
    hover_name="City",                # Mostra la città al passaggio del mouse
    animation_frame="Day",         # CREA LA BARRA SCORREVOLE PER GIORNI
    scope='usa',                      # Inquadra solo gli Stati Uniti
    color_continuous_scale="Reds",    # Colore rosso per enfatizzare il pericolo
    range_color=[0, df_agosto_2017['Precipitation(mm)'].max()],
    title="Spostamento e Intensità dell'Uragano Harvey (Agosto 2017)",
    height=700,
    template="plotly_dark"            # Sfondo scuro per far risaltare i punti
)


fig.show()

## Tempesta invernale Uri (Febbraio 2021)

In [ ]:
df['StartTime(UTC)'] = pd.to_datetime(df['StartTime(UTC)'])
df['Day'] = df['StartTime(UTC)'].dt.day

df_febbraio_2021 = df[(df['StartTime(UTC)'].dt.year == 2021) & (df['StartTime(UTC)'].dt.month == 2) & (df['Type'] == 'Snow') & (df['Severity'] == 'Heavy')]

df_febbraio_2021['Precipitation(mm)'] = df_febbraio_2021['Precipitation(in)'] *25.4

df_febbraio_2021 = df_febbraio_2021[df_febbraio_2021['Precipitation(mm)'] < 500]

df_febbraio_2021 = df_febbraio_2021.sort_values(by='Day')


fig = px.scatter_geo(
    df_febbraio_2021,
    lat="LocationLat",
    lon="LocationLng",
    color="Precipitation(mm)",        # Il colore indica l'intensità
    size="Precipitation(mm)",         # La dimensione della bolla indica l'intensità
    hover_name="City",                # Mostra la città al passaggio del mouse
    animation_frame="Day",         # CREA LA BARRA SCORREVOLE PER GIORNI
    scope='usa',                      # Inquadra solo gli Stati Uniti
    color_continuous_scale="Blues",    # Colore rosso per enfatizzare il pericolo
    range_color=[0, df_febbraio_2021['Precipitation(mm)'].max()],
    title="Spostamento e Intensità delle tempeste invernali del 2021",
    height=700,
    template="plotly_dark"            # Sfondo scuro per far risaltare i punti
)

fig.show()


In [ ]:
df['StartTime(UTC)'] = pd.to_datetime(df['StartTime(UTC)'])
df['Year'] = df['StartTime(UTC)'].dt.year

df_clean = df.copy()
df_clean['Precipitation(mm)'] = df_clean['Precipitation(in)'] * 25.4
df_clean = df_clean[(df_clean['Precipitation(mm)'] > 0) & (df_clean['Precipitation(mm)'] < 500)]

precip_totali = df_clean.groupby(['Year', 'State'])['Precipitation(mm)'].sum().reset_index(name='Totale_Precipitazioni')

df_snow = df_clean[df_clean['Type'] == 'Snow']
precip_neve = df_snow.groupby(['Year', 'State'])['Precipitation(mm)'].sum().reset_index(name='Precipitazioni_Neve')

df_merged = pd.merge(precip_totali, precip_neve, on=['Year', 'State'], how='left')
df_merged['Precipitazioni_Neve'] = df_merged['Precipitazioni_Neve'].fillna(0)

df_merged['Percentuale_Neve'] = (df_merged['Precipitazioni_Neve'] / df_merged['Totale_Precipitazioni']) * 100

df_merged = df_merged.sort_values(by='Year')

fig = px.choropleth(
    df_merged,
    locations='State', 
    locationmode="USA-states", 
    color='Percentuale_Neve',
    animation_frame='Year', 
    scope="usa", 
    color_continuous_scale="Blues",
    range_color=[0, 100], 
    title="Percentuale di Precipitazioni Nevose per Stato (per Anno)",
    labels={'Percentuale_Neve': '% Neve sul Totale', 'State': 'Stato'},
    template="plotly_dark"
)

fig.layout.updatemenus[0].buttons[0].args[1]['frame']['duration'] = 1200 

fig.show()

In [ ]:


df['StartTime(UTC)'] = pd.to_datetime(df['StartTime(UTC)'])
df['Date'] = df['StartTime(UTC)'].dt.date
df['Year'] = df['StartTime(UTC)'].dt.year

df_snow = df[df['Type'] == 'Snow'].copy()
df_snow['Precipitation(mm)'] = df_snow['Precipitation(in)'] * 25.4

df_snow = df_snow[(df_snow['Precipitation(mm)'] > 0) & (df_snow['Precipitation(mm)'] < 500)]

idx_max_daily = df_snow.groupby('Date')['Precipitation(mm)'].idxmax()
df_snow_daily_max = df_snow.loc[idx_max_daily].copy()

df_snow_daily_max = df_snow_daily_max.sort_values(by='Year')

fig = px.scatter_geo(
    df_snow_daily_max,
    lat="LocationLat",
    lon="LocationLng",
    color="Precipitation(mm)",        
    size="Precipitation(mm)",         
    hover_name="City",                
    animation_frame="Year",         
    scope='usa',                      
    color_continuous_scale="Blues",    
    range_color=[0, df_snow_daily_max['Precipitation(mm)'].max()],
    title="Picco Massimo Giornaliero di Neve negli Stati Uniti (per Anno)",
    height=700,
    template="plotly_dark",
    size_max=30
)

fig.layout.updatemenus[0].buttons[0].args[1]['frame']['duration'] = 1200 
fig.update_geos(
    lataxis_range=[24, 50],  
    lonaxis_range=[-125, -65]
)

fig.show()

In [ ]:
df['StartTime(UTC)'] = pd.to_datetime(df['StartTime(UTC)'])
df['Year'] = df['StartTime(UTC)'].dt.year

df_filtered = df[df['Type'] != 'Precipitation']

df_counts = df_filtered.groupby(['Year', 'Type']).size().reset_index(name='Count')
df_counts = df_counts.sort_values(by=['Year', 'Count'])

max_count = df_counts['Count'].max()

fig = px.bar(
    df_counts,
    x='Count',
    y='Type',
    animation_frame='Year',       
    orientation='h',              
    title="Numero Totale di Eventi per Tipo (per Anno)",
    labels={'Count': 'Numero Totale', 'Type': 'Tipo di Evento'},
    range_x=[0, max_count + (max_count * 0.1)], 
    color='Type',                 
)

fig.update_yaxes(categoryorder='total ascending')
fig.layout.updatemenus[0].buttons[0].args[1]['frame']['duration'] = 1200 
fig.update_layout(legend_traceorder="reversed")

fig.show()

In [ ]:
df['StartTime(UTC)'] = pd.to_datetime(df['StartTime(UTC)'])
df['Year'] = df['StartTime(UTC)'].dt.year

df_filtered = df[df['Type'] != 'Precipitation']

df_counts = df_filtered.groupby(['Year', 'Type']).size().reset_index(name='Count')
df_counts = df_counts.sort_values(by=['Year', 'Count'])

max_count = df_counts['Count'].max()

fig = px.bar(
    df_counts,
    x='Count',
    y='Type',
    animation_frame='Year',       
    orientation='h',              
    title="Numero Totale di Eventi per Tipo (per Anno) - Scala Logaritmica",
    labels={'Count': 'Numero Totale (Scala Log)', 'Type': 'Tipo di Evento'},
    log_x=True,                   # Abilita l'asse X logaritmico
    range_x=[1, max_count * 1.5], # Imposta il range partendo da 1 (il logaritmo di 0 non è definito)
    color='Type',                 
)

fig.update_yaxes(categoryorder='total ascending')
fig.layout.updatemenus[0].buttons[0].args[1]['frame']['duration'] = 1200 
fig.update_layout(legend_traceorder="reversed")

fig.show()